In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Plotting code
The following code block includes a helper function for plotting a histrogram.
I've hidden it for the purposes of discussion in class but feel free to take a look when editing this notebook.

In [ ]:
# @title
def create_probability_bar_plot(probabilities, point = None, title='Probability Bar Plot', xlabel='Position', ylabel='Probabilities', color='blue'):
    """
    Creates a bar plot of probabilities using Matplotlib.

    Args:
        probabilities (numpy.array): A 1D numpy array of probabilities.
        title (str): The title of the plot.
        xlabel (str): The label for the x-axis.
        ylabel (str): The label for the y-axis.
        color (str): The color of the bars.
    """

    # Create an array of indices to represent the categories
    indices = np.arange(len(probabilities))

    # Create the bar plot
    plt.bar(indices, probabilities, color=color)

    # Add an indicator for the current state
    if point is not None:
      plt.scatter(point,probabilities[point], color='orange')
    # Set the title and axis labels
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)

    # Set the x-axis ticks and labels
    plt.xticks(indices)

    # Show the plot
    plt.show()

In [ ]:
def normalize(belief):
    s = np.sum(belief)
    return belief / s

def discrete_bayes(likelihood, belief):
  belief = likelihood * belief # P(z|x)*bel_prior(x)
  belief = normalize(belief) # Normalize
  return belief

def lh_hallway( z, map, z_prob):
    """ compute likelihood that a measurement matches
    positions in the hallway."""

    try:
        scale = z_prob / (1. - z_prob)
    except ZeroDivisionError:
        scale = 1e8
    likelihood = np.ones(len(map))
    likelihood[np.array(map)==z] *= scale
    likelihood /= sum(likelihood) # Normalize
    return likelihood

hallway = np.array([1, 1, 0, 0, 0, 0, 0, 0, 1, 0])
belief = np.array([0.1] * 10)
likelihood = lh_hallway(z=1, map=hallway, z_prob=.75)
belief = discrete_bayes(likelihood=likelihood, belief=belief)
create_probability_bar_plot(belief)

In [ ]:
likelihood = lh_hallway(z=1, map=hallway, z_prob=.75)
belief = discrete_bayes(likelihood=likelihood, belief=belief)
create_probability_bar_plot(belief)

# The Bayes Filter
This is the main simulation function that executes all the steps of the discrete bayes algorithm.

For the sake of simplicity the simulated component is deterministic even if the estimation is probablisitic.

1. Update the true state of the system and take a measurement in the background
2. Apply the motion model to create the prior belief
3. Apply the sensor model to compute the likelihood of the measurement for each state
4. Use the bayes filter equation to correct the belief.

In [ ]:
def bayes_simulator_1d(x, belief, map, motion_model, sensor_model, action=1):
  # Apply motion to the true robot (for simplicity it is deterministic)
  x = ( x + action ) % len(map)
  # Take a measurement  (for simplicity it is deterministic)
  z = map[x]
  # Update the prior P(x_t| x_t-1, u)
  prior = motion_model(belief, action=action)
  # Compute the likelihood of the measurement P(z|x,map)
  likelihood = sensor_model(z, map)
  # Correct the belief states based on the measurement
  posterior = discrete_bayes(likelihood, prior)
  return x, posterior

## Our interative widget helper

`make_simulator_plot` is a helpful function for creating an interactive widget. This allows us to see how the 1D discrete Bayes filter evolves over time given different maps, motion models, and sensor models

In [ ]:
from ipywidgets import interact, IntSlider

def make_simulator_plot(x, map, belief, motion_model, sensor_model, timesteps=20):
  beliefs = [belief]
  states = [x]
  for _ in range(timesteps):
    x, belief = bayes_simulator_1d(x, belief, map, motion_model, sensor_model)
    beliefs.append(belief)
    states.append(x)
  def simulator(time_step):
    create_probability_bar_plot(beliefs[time_step], states[time_step])
    plt.show()
  return simulator

In [ ]:
def perfect_predict(belief, action=1):
    """ A perfect motion model with no error in the action execution
    """
    n = len(belief)
    result = np.zeros(n)
    for i in range(n):
        # Perfect motion means P(x|u) = 1
        P_x_u = 1
        P_x_t_minus_1 = belief[(i-action) % n]
        # P(x) = P(x|u) * P(x_t-1)
        result[i] =  1 * P_x_t_minus_1
    return result

def no_measurement_model(z, map):
    """ A likelihood function that ignore the actual sensor measurement
        To represent a "blind" system or one with no correction step
    """
    return np.ones(len(map)) / len(map)

In [ ]:
belief = np.array([.35, .1, .2, .3, 0, 0, 0, 0, 0, .05])
perfect_beliefs = []

timesteps = 20
simulate = make_simulator_plot(
    x=3,
    map=[1, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    belief=belief,
    motion_model=perfect_predict,
    sensor_model=no_measurement_model,
    timesteps=20)

interact(simulate, time_step=IntSlider(value=0, max=timesteps));

In [ ]:
def simple_motion_model(belief, action = 1, p_under = 0.1, p_correct = 0.8, p_over = 0.1):
    n = len(belief)
    prior = np.zeros(n)
    for i in range(n):
      # Sum(P(x|u) * P(z|x)) for x_t-1
        prior[i] = (
            belief[(i-action) % n]   * p_correct +
            belief[(i-action-1) % n] * p_over +
            belief[(i-action+1) % n] * p_under)
    return prior

belief = [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]
timesteps = 20
simulate = make_simulator_plot(
    x=3,
    map=[1, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    belief=belief,
    motion_model=simple_motion_model,
    sensor_model=no_measurement_model,
    timesteps=20)

interact(simulate, time_step=IntSlider(value=0, max=timesteps-1));

In [ ]:
def predict_move_convolution(belief, action = 1, kernel = [0.1, 0.8, 0.1]):
    N = len(belief)
    kN = len(kernel)
    width = int((kN - 1) / 2)

    prior = np.zeros(N)
    for i in range(N):
        for k in range (kN):
            index = (i + (width-k) - action) % N
            prior[i] += belief[index] * kernel[k]
    return prior

In [ ]:
belief = [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]
timesteps = 20
simulate = make_simulator_plot(
    x=3,
    map=[1, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    belief=belief,
    motion_model=lambda x, action: predict_move_convolution(x, action, kernel=[0.1,0.2,0.4,0.2,0.1]),
    sensor_model=no_measurement_model,
    timesteps=20)

interact(simulate, time_step=IntSlider(value=0, max=timesteps-1));

In [ ]:
belief = np.array([.1] * 10)
timesteps = 20
simulate = make_simulator_plot(
    x=0,
    map=[1, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    belief=belief,
    motion_model=lambda x, action: predict_move_convolution(x, action, [0.1,0.8,0.1]),
    sensor_model=lambda z, map: lh_hallway(z, map, z_prob=0.9),
    timesteps=20)

interact(simulate, time_step=IntSlider(value=0, max=timesteps));